<a href="https://colab.research.google.com/github/VasilievNichita/IA_LABS/blob/main/Lab_2_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install kagglehub scikit-learn pandas numpy matplotlib

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

from sklearn.decomposition import PCA

In [ ]:
path = kagglehub.dataset_download("heptapod/titanic")
path

In [ ]:
os.listdir(path)

In [ ]:
csvs = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
csvs

In [ ]:
train_path = [f for f in csvs if os.path.basename(f).lower() == "train.csv"][0]
train_path

In [ ]:
df = pd.read_csv(train_path)
df.head()

In [ ]:
df.info()

In [ ]:
df["Survived"].value_counts()

In [ ]:
target = "Survived"
drop_cols = ["Name", "Ticket", "Cabin"]

X = df.drop(columns=[target] + [c for c in drop_cols if c in df.columns])
y = df[target].astype(int)

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

num_cols, cat_cols

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape, X_test.shape

In [ ]:
def evaluate_model(name, pipeline):
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)

    print("="*70)
    print("MODEL:", name)
    print("Accuracy:", round(accuracy_score(y_test, pred), 4))
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))
    print("Report:\n", classification_report(y_test, pred, digits=4))

In [ ]:
nb_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("to_dense", FunctionTransformer(lambda x: x.toarray() if hasattr(x, "toarray") else x)),
    ("model", GaussianNB())
])

svm_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", SVC(kernel="rbf", C=2.0, gamma="scale", random_state=42))
])

tree_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", DecisionTreeClassifier(max_depth=5, random_state=42))
])

ada_pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", AdaBoostClassifier(n_estimators=300, learning_rate=0.5, random_state=42))
])

In [ ]:
evaluate_model("Bayes: GaussianNB", nb_pipeline)
evaluate_model("Functional: SVM (RBF)", svm_pipeline)
evaluate_model("Trees: DecisionTree (depth=5)", tree_pipeline)
evaluate_model("Meta: AdaBoost", ada_pipeline)

In [ ]:
evaluate_model("Bayes: GaussianNB", nb_pipeline)
evaluate_model("Functional: SVM (RBF)", svm_pipeline)
evaluate_model("Trees: DecisionTree (depth=5)", tree_pipeline)
evaluate_model("Meta: AdaBoost", ada_pipeline)

In [ ]:
def plot_regions(X2d, y, clf, title):
    x_min, x_max = X2d[:, 0].min() - 1, X2d[:, 0].max() + 1
    y_min, y_max = X2d[:, 1].min() - 1, X2d[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 400),
        np.linspace(y_min, y_max, 400)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = clf.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, Z, alpha=0.25)
    plt.scatter(X2d[:, 0], X2d[:, 1], c=y, s=20, edgecolor="k", alpha=0.9)
    plt.title(title)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.show()

In [ ]:
models_2d = {
    "Bayes (2D)": GaussianNB(),
    "SVM RBF (2D)": SVC(kernel="rbf", C=2.0, gamma="scale", random_state=42),
    "DecisionTree (2D)": DecisionTreeClassifier(max_depth=5, random_state=42),
    "AdaBoost (2D)": AdaBoostClassifier(n_estimators=300, learning_rate=0.5, random_state=42),
}

for name, clf in models_2d.items():
    clf.fit(Xtr_2d, y_train)
    plot_regions(Xtr_2d, y_train, clf, f"{name} — TRAIN")
    plot_regions(Xte_2d, y_test, clf, f"{name} — TEST")